# Mapping & joining values.

**Python Foundation · An interactive experiment · 05**

Choose **Runtime → Run all** in Google Colab. Then edit the list, choose a transformation, and change the joining separator in the explorer. Press Enter after editing a text field to update it. All code cells are editable and execute real Python.

`map(function, iterable)` returns an iterator that applies a function as values are requested. `separator.join(strings)` combines strings into one string, inserting the separator between adjacent items.


In [ ]:
digits = [1, 2, 9]
strings = list(map(str, digits))
joined = "".join(strings)
number = int(joined)

print("Original list:", digits)
print("After map(str, ...):", strings)
print("After join:", repr(joined), type(joined).__name__)
print("After int:", number, type(number).__name__)
print("With a separator:", "-".join(strings))


## Trace the pipeline

| Step | Result | Type |
|---|---|---|
| `digits` | `[1, 2, 9]` | `list` of integers |
| `map(str, digits)` | Iterator, not yet a list | `map` |
| `list(map(str, digits))` | `['1', '2', '9']` | `list` of strings |
| `''.join(map(str, digits))` | `'129'` | `str` |
| `int(''.join(map(str, digits)))` | `129` | `int` |

Read `int("".join(map(str, digits)))` from the inside out: convert each item to a string, join the strings, then parse the whole string as an integer. `join` accepts an iterable of strings, so the intermediate `list(...)` is optional.

For the digit-list interpretation, each item should be an integer from 0 to 9. Parsing as `int` removes leading zeros; an empty joined string cannot be parsed as an integer.


In [ ]:
print("Direct iterator:", "".join(map(str, digits)))
print("Leading zero:", repr("".join(map(str, [0, 1, 2]))))
print("As an integer:", int("".join(map(str, [0, 1, 2]))))
print("Empty iterable:", repr("-".join([])))

try:
    print("".join(digits))
except TypeError as error:
    print("Expected TypeError:", error)

try:
    print(int("".join(map(str, []))))
except ValueError as error:
    print("Expected ValueError:", error)


## Lazy evaluation and one-way consumption

Creating a `map` object does not call the transformation on every item immediately. `next`, `list`, and `join` consume it. Once consumed, the same iterator does not restart; create a new `map` to repeat the operation. A stored list can be reused.


In [ ]:
def traced_str(value):
    print("Transforming:", value)
    return str(value)

mapped = map(traced_str, digits)
print("Created the map; no items transformed yet.")
print("First item:", next(mapped))
print("Remaining items:", list(mapped))
print("Already exhausted:", list(mapped))
print("Fresh iterator:", list(map(str, digits)))


## Explore map → join → int

Enter a Python list of literal values, such as `[1, 2, 9]`, `[0, 1, 2]`, or `['hello', 'world']`. The explorer accepts literal lists rather than arbitrary Python expressions. Choose `str`, `repr`, or a transformation that doubles each item. It shows the transformed values, their types, whether `join` succeeds, and whether the joined string can be parsed as an integer.

Try doubling the numbers: `map` can return integers, but `join` needs strings. Change the separator to `-` and observe why `'1-2-9'` is not an integer literal.


In [ ]:
import ast
import html
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

def panel(rows, note=""):
    escape = lambda value: html.escape(str(value))
    body = "".join(
        f'<tr><td style="padding:8px;vertical-align:top">{escape(label)}</td>'
        f'<td style="padding:8px;overflow-wrap:anywhere"><code>{escape(value)}</code></td></tr>'
        for label, value in rows
    )
    return (f'<div style="padding:20px;border:1px solid #cbd5e1;border-radius:14px;'
            f'background:#fff;color:#172b3a;font:15px/1.7 system-ui">'
            f'<table style="width:100%;text-align:left">{body}</table><p>{escape(note)}</p></div>')


transforms = {"str": str, "repr": repr, "double each item": lambda value: value * 2}

def mapping_report(source, transform_name, separator):
    try:
        values = ast.literal_eval(source)
        if not isinstance(values, list):
            raise ValueError("Enter a list such as [1, 2, 9].")
    except (ValueError, SyntaxError, TypeError) as error:
        return panel([("Input error", str(error))], "Use a Python list of literal values.")
    rows = [("Input list", repr(values)), ("Function", transform_name)]
    try:
        mapped = map(transforms[transform_name], values)
        rows.append(("map object type", type(mapped).__name__))
        converted = list(mapped)
        rows.extend([("Transformed list", repr(converted)),
                     ("Item types", repr([type(value).__name__ for value in converted])),
                     ("Read the same iterator again", repr(list(mapped))),
                     ("Separator", repr(separator))])
    except (TypeError, ValueError) as error:
        rows.append((type(error).__name__, str(error)))
        return panel(rows, "The transformation failed while consuming the map iterator.")
    try:
        joined = separator.join(converted)
        rows.append(("Joined string", repr(joined)))
    except TypeError as error:
        rows.append(("join → TypeError", str(error)))
        return panel(rows, "Every item passed to join must be a string. Try the str transformation.")
    try:
        rows.append(("int(joined)", repr(int(joined))))
    except ValueError as error:
        rows.append(("int → ValueError", str(error)))
    return panel(rows, "map transforms items; join combines strings; int parses the complete string. The original list is unchanged.")

values_input = widgets.Text(value="[1, 2, 9]", description="List:", continuous_update=False, layout=widgets.Layout(width="95%"))
transform_input = widgets.Dropdown(options=list(transforms), value="str", description="Function:")
separator_input = widgets.Text(value="", description="Separator:", continuous_update=False, layout=widgets.Layout(width="95%"))
mapping_view = widgets.HTML()
mapping_reset = widgets.Button(description="Reset")

def update_mapping(change=None):
    mapping_view.value = mapping_report(values_input.value, transform_input.value, separator_input.value)

def reset_mapping(button):
    values_input.value = "[1, 2, 9]"
    transform_input.value = "str"
    separator_input.value = ""
    update_mapping()

for control in (values_input, transform_input, separator_input):
    control.observe(update_mapping, names="value")
mapping_reset.on_click(reset_mapping)
update_mapping()
display(widgets.VBox([values_input, transform_input, separator_input, mapping_reset, mapping_view]))


## Practice: Plus One

The string-conversion approach builds a number, adds one, then converts its digits back to a list. Edit `digits` below and rerun the cell. Try `[9, 9, 9]`.

This is a learning example using integer conversion. The digit-by-digit carry approach is useful when a task restricts whole-number conversion or when you want to study carry handling.

[LeetCode 66 — two approaches and complexity notes](https://github.com/Lawson-Dong/leetcode-solution/tree/main/easy/066)


In [ ]:
digits = [1, 2, 9]  # Try [9, 9, 9].
number = int("".join(map(str, digits)))
updated = number + 1
answer = list(map(int, str(updated)))
print("Original:", digits)
print("Number:", number)
print("After adding one:", updated)
print("Result:", answer)


## Remember

- `map` transforms one item at a time and returns a consumable iterator.
- `list(map(...))` collects the results into a reusable list.
- `separator.join(...)` combines strings and returns a new string.
- `int(...)` parses the complete string; it does not convert each character separately.

References: [Python map](https://docs.python.org/3/library/functions.html#map), [Python str.join](https://docs.python.org/3/library/stdtypes.html#str.join).
